# Simulationsgestützte Optimierung mit SciPy

In den nachfolgenden Aufgaben werden Sie mithilfe von scipy.optimize und Ihrem entwickelten Simulationsmodell das Energiesystem eines Gebäudes optimieren. Sie können die Aufgaben entweder direkt hier im Jupyter-Notebook ausarbeiten oder in einem eigenen Python-Skript (z.B. mit Spyder).

## Aufgabe 0

Instanzieren und testen Sie Ihr entwickeltes Simulationsmodell. Verwenden sie dafür folgende Parameterwerte:
* battery_kWh =  300
* pv_kWp = 192

Für die Trucks (und die übrigen Parameter) verwenden Sie die auch bisher verwendeten Einstellungen.
Testen Sie die Funktionsfähigkeit der Simulation, indem Sie die Funktion simulate() aufrufen. Lassen Sie sich die berechneten Gesamtkosten (system costs + operating costs über 10 Jahre) ausgeben. (0 %)

In [1]:
import sys
sys.path.append('../Simulation')

from Model_solution import *
from scipy.optimize import minimize, differential_evolution
import time


trucks = [Etruck("workday_lunchbreak") for i in range(7)]
    #trucks+= [Etruck("worknight") for i in range(2)]
    #trucks+= [t1 for i in range(10)]
    #trucks+= [t2 for i in range(10)]

results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=300, 
                   pv_kWp=192,
                   #monthly_km=[68000,61000,66000,
                   #                63000,65000,64000,
                   #                64000,62000,60000,
                   #                68000,65000,68000,],
                   grid_threshold=0.99,
                   )
print(results)
print(f'Total cost: {results.system_cost + results.operating_cost*10:.0f} €/10a')

Energy Flows: 
Grid                346438.0
PV to Truck          48780.0
PV to Battery        69685.0
PV to Grid           80898.0
Battery to Truck     69669.0
Grid to Truck       346438.0
Driven             -548100.0
dtype: float64
PV Yield: 199363 kWh/a
____________________
Total cost: 2_053_075 €/10a
self.system_cost=378_000.0€
self.operating_cost=167_507.5€/a
self.emissions=93538.2kg/a
____________________
Self-consumption: 59%
Load Cycles: 232.3
Total cost: 2053075 €/10a


## Aufgabe 1

Definieren Sie eine Funktion *objective_function_PV(x)*, welche als Argument *x* die Leistung (kWp) der PV-Anlage erhält, den Wert entsprechend im Simulationsmodell setzt, die Simulation ausführt und die berechneten Gesamtkosten (system costs + operating costs über 10 Jahre) als Ergebnis zurückliefert. Der Parameter battery_kWh = 300 soll dabei unverändert bleiben. (10 %)

In [67]:
def objective_function_PV(x):
    results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=300, 
                   pv_kWp=x,
                   #monthly_km=[68000,61000,66000,
                   #            63000,65000,64000,
                   #            64000,62000,60000,
                   #            68000,65000,68000,],
                   grid_threshold=0.99,
                   )
    return (results.system_cost + results.operating_cost*10)

print(objective_function_PV(1))

2415610.6416863953


## Aufgabe 2

Programmieren Sie eine einfache Grid Search (in 1D), indem Sie in einer for-Schleife die oben definierte Funktion *objective_function_PV()* für verschiedene kWp-Werte (im Bereich 50-800 kW in 50 kW-Schritten) aufrufen und die Rückgabewerte ausgeben (Funktion *print*). Bei welchem dieser kWp-Werte sind die berechneten Gesamtkosten am geringsten? (10 %)

In [6]:
for i in range(5,801,50):
    r = objective_function_PV(i)
    print(f"{i} kWp, Cost: {r}")

5 kWp, Cost: 2400829.872109375
55 kWp, Cost: 2237494.426083969
105 kWp, Cost: 2137279.3079833435
155 kWp, Cost: 2083632.9640065623
205 kWp, Cost: 2050183.0928961562
255 kWp, Cost: 2027703.2375960313
305 kWp, Cost: 2018914.2281329688
355 kWp, Cost: 2016283.6640337496
405 kWp, Cost: 2016332.5979352186
455 kWp, Cost: 2021547.9065125627
505 kWp, Cost: 2027199.3346869063
555 kWp, Cost: 2034843.954908
605 kWp, Cost: 2045078.604821844
655 kWp, Cost: 2057743.0350321871
705 kWp, Cost: 2073057.308074094
755 kWp, Cost: 2090951.4730773126


## Aufgabe 3

Verwenden Sie die Funktion *scipy.optimize.minimize()* aus dem Python-Package *scipy* um den optimalen kWp-Wert für die PV-Anlage zu finden, der die Gesamtkosten minimiert. Verwenden Sie die oben definierte Funktion *objective_function_PV()* als Zielfunktion und legen Sie geeignete Grenzwerte (*bounds*) sowie einen sinnvollen Startwert *x0* fest.

(Hinweis: Unter Umständen kann der Rechenvorgang ein paar Minuten dauern.)

Falls Sie mit der Default-Methode Probleme haben, testen Sie auch andere Optimierungsverfahren (speziell 'trust-constr' oder auch 'CG'), indem Sie die Funktion *minimize* mit der ensprechenden Option *method='trust-constr'* bzw. method='CG' aufrufen. Ggf. kann es auch helfen die Toleranzen enger zu setzen (z.B. Option *tol=1e-6*). Nähere Details dazu können Sie in der Hilfe nachlesen.

Lassen Sie sich den Rückgabewert von *minimize()* in der Konsole ausgeben. Wie lautet der berechnete optimale kWp-Wert? (15 %)

In [78]:
bounds = [(200,500)]
#options={'gtol': 1e-16, 'eps': 1e-20, 'ftol': 2e-16}

res = minimize(
    objective_function_PV,
    x0 = 300,
    bounds = bounds,
    #method = "Nelder-Mead",
    #method = 'CG',
    #method = 'L-BFGS-B',
    method = 'trust-constr',
    #options = options
    #tol=1e-6
    )
print("done.")
res

c:\Users\heinzl\Anaconda3\lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '
c:\Users\heinzl\Anaconda3\lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '
c:\Users\heinzl\Anaconda3\lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quas

done.


 barrier_parameter: 2.048000000000001e-09
 barrier_tolerance: 2.048000000000001e-09
          cg_niter: 374
      cg_stop_cond: 2
            constr: [array([381.92368921])]
       constr_nfev: [0]
       constr_nhev: [0]
       constr_njev: [0]
    constr_penalty: 1.0
  constr_violation: 0.0
    execution_time: 502.1965136528015
               fun: array([2015670.89061597])
              grad: array([12519.42450215])
               jac: [<1x1 sparse matrix of type '<class 'numpy.float64'>'
	with 1 stored elements in Compressed Sparse Row format>]
   lagrangian_grad: array([12518.14839502])
           message: '`xtol` termination condition is satisfied.'
            method: 'tr_interior_point'
              nfev: 772
              nhev: 0
               nit: 386
             niter: 386
              njev: 386
        optimality: 12518.148395015953
            status: 2
           success: True
         tr_radius: array([3.24797884e-09])
                 v: [array([-1.27610713])]
      

In [45]:
res4 = differential_evolution(
    objective_function_PV,
    bounds=bounds,
    popsize=20,
    mutation=0.05,
    recombination=0.05)
print("done.")
res4

done.


     fun: 2015671.2003697276
 message: 'Optimization terminated successfully.'
    nfev: 80
     nit: 1
 success: True
       x: array([381.99592393])

## Aufgabe 4

Definieren Sie eine zweite Funktion *objective_function_2D(x)*, welche als Inputwert x einen Vektor (Liste) aus den beiden Parametern
* Dimenstionierung (kWp) der PV-Anlage
* Batteriekapazität

erhält, die Parameter entsprechend im Simulationsmodell setzt, die Simulation ausführt und die berechneten Gesamtkosten (system costs + operating costs über 10 Jahre) als Ergebnis zurückliefert. (10 %)

In [79]:
def objective_function_2D(x):
    results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=x[1], 
                   pv_kWp=x[0],
                   #monthly_km=[68000,61000,66000,
                   #            63000,65000,64000,
                   #            64000,62000,60000,
                   #            68000,65000,68000,],
                   grid_threshold=0.99,
                   )
    return (results.system_cost + results.operating_cost*10)

print(objective_function_2D((5,10)))

2316583.543348568


## Aufgabe 5

Verwenden Sie wieder die Funktion *scipy.optimize.minimize()* aus dem Package scipy um sowohl den kWp-Wert für die PV-Anlage als auch die Batteriekapzazität gleichzeitig zu optimieren. Verwenden Sie dazu die oben definierte Funktion *objective_function_2D()* als Zielfunktion und legen Sie geeignete Grenzwerte (*bounds*) sowie sinnvolle Startwerte *x0* fest.
Testen Sie auch andere Optimierungsverfahren (speziell 'trust-constr') und vergleichen Sie die Ergebnisse. (15 %)

In [89]:
bounds2 = [(200,800), (400,900)]
#options={'gtol': 1e-6, 'eps': 1e-12, 'ftol': 2e-9}

res2 = minimize(
    objective_function_2D,
    x0 =(400,700),
    bounds = bounds2,
    #method = "Nelder-Mead",
    #method = 'CG',
    method = 'trust-constr'
    #options = options
    )
print("done.")
res2

done.


 barrier_parameter: 2.048000000000001e-09
 barrier_tolerance: 2.048000000000001e-09
          cg_niter: 785
      cg_stop_cond: 2
            constr: [array([429.80323257, 762.15889663])]
       constr_nfev: [0]
       constr_nhev: [0]
       constr_njev: [0]
    constr_penalty: 1.0
  constr_violation: 0.0
    execution_time: 1349.7353127002716
               fun: 1859273.506798511
              grad: array([2.06184465, 3.71399751])
               jac: [<2x2 sparse matrix of type '<class 'numpy.float64'>'
	with 2 stored elements in Compressed Sparse Row format>]
   lagrangian_grad: array([2.06179057, 3.71377374])
           message: '`xtol` termination condition is satisfied.'
            method: 'tr_interior_point'
              nfev: 1959
              nhev: 0
               nit: 662
             niter: 662
              njev: 653
        optimality: 3.7137737386599734
            status: 2
           success: True
         tr_radius: 6.248283771690705e-09
                 v: [array(

## Aufgabe 6

Verwenden Sie anstelle von *minimize()* die Funktion *scipy.optimize.differential_evolution()* um das Optimum mittels Differential-Evolution-Verfahren zu berechnen. Legen Sie dazu wieder geeignete Grenzen (*bounds*) fest. Vergleichen Sie die Ergebnisse mit jenen aus Aufgabe 5.
> Hinweis: Wie genau die Funktion differential_evolution aufgerufen werden kann (und welche Input-Argumente sie erlaubt bzw. benötigt), können Sie in der Dokumentation nachlesen: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html

Probieren Sie auch verschiedene Verfahrensparameter aus, speziell die Populationsgröße (*popsize*) sowie Mutations- und Rekombinationsraten (*mutation*, *recombination*). (15 %)

In [88]:
res3 = differential_evolution(
    objective_function_2D,
    bounds=[(200,1000), (200,1000)],
    popsize=20,
    mutation=0.05,
    recombination=0.05,
    tol=1e-3
    )
print("done.")
res3

done.


     fun: 1859287.7952736423
     jac: array([-2.49128591, -0.58207615])
 message: 'Optimization terminated successfully.'
    nfev: 331
     nit: 6
 success: True
       x: array([427.24103746, 757.54391839])

## Aufgabe 7

Vergleichen Sie die Rechenzeit sowie die Anzahl der Funktionsauswertungen 
* zwischen *minimize()* im 1D-Fall (Aufgabe 3) und im 2D-Fall (Aufgabe 5),
* zwischen *minimize()* (Aufgabe 5) und *differential_evolution()* (Aufgabe 6), 

indem die Werte jeweils in einer Tabelle gegenüberstellen. Welches der Verfahren ist im jeweiligen Fall effizienter? (10 %)
> Hinweis: Die Anzahl der Funktionsauswertungen (*nfev*) wird Ihnen von den beiden Funktionen jeweils mit zurück geliefert. Um die Rechenzeit zu bestimmen, vergleichen Sie die Zeiten vor und nach dem Funktionsaufruf. Verwenden Sie dazu das Python-Package *time*:  
>> *import time  
>> t = time.time()  
>> (...Funktionsaufruf...)  
>> elapsed_time = time.time() - t*  



In [ ]:
import time

# 1D
t = time.time()
res = minimize(objective_function_PV, x0 = 300, bounds = bounds, method = 'trust-constr')
elapsed_time_1D = time.time() - t
print("done.")

In [ ]:
# 2D
t = time.time()
res2 = minimize(objective_function_2D, x0 = (300,300), bounds = bounds2, method = 'trust-constr')
elapsed_time_2D = time.time() - t
print("done.")

In [ ]:
# DE
t = time.time()
res3 = differential_evolution(objective_function_2D, bounds=bounds2, popsize=20, mutation=0.05, recombination=0.05)
elapsed_time_DE = time.time() - t
print("done.")

In [ ]:
from tabulate import tabulate
headers = [' ', 'runtime', 'nfev']
tab1 = [['1D', elapsed_time_1D, res.nfev],['2D', elapsed_time_2D, res2.nfev]]
tab2 = [['2D', elapsed_time_2D, res2.nfev],['DE', elapsed_time_DE, res3.nfev]]

print(tabulate(tab1, headers=headers))
print(' ')
print(tabulate(tab2, headers=headers))

## Aufgabe 8

Variieren Sie andere Parameterwerte aus der Simulation, speziell
* Energiepreise,
* Trucks Anzahl und schedules,

in sinnvollen Bereichen per Hand und führen Sie die Optimierungsläufe von Aufgabe 5 nochmals durch. Wie stark ändern sich die optimalen PV- und Batteriewerte?  
(15 %)